In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers sentencepiece accelerate -q

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH = f"{DRIVE_BASE}/pubmedqa_filtered.json"
CHECKPOINT_DIR = f"{DRIVE_BASE}/tuned_lens_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)
print(f"Corpus loaded: {len(corpus)} samples")

In [ ]:
MODEL_ID = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="eager"
)
model = model.to("cuda")
model.eval()

for param in model.parameters():
    param.requires_grad = False

n_layers = model.config.n_layer
hidden_size = model.config.n_embd
vocab_size = model.config.vocab_size

print(f"Model loaded on GPU: {n_layers} layers, hidden size {hidden_size}")
print(f"Model parameters frozen: {sum(p.requires_grad for p in model.parameters())} trainable")

In [ ]:
class TunedLensTranslator(nn.Module):
    """
    Affine translator for a single transformer layer.
    Maps intermediate hidden state to a corrected vocabulary distribution.

    Trained to minimize KL divergence between:
    - Its projection of layer i hidden state
    - The final layer's output distribution (ground truth)

    Initialized as identity so it starts as raw logit lens
    and learns corrections from there.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.translator = nn.Linear(hidden_size, hidden_size, bias=True)
        nn.init.eye_(self.translator.weight)
        nn.init.zeros_(self.translator.bias)

    def forward(self, hidden_state):
        h = hidden_state.float()
        translated = self.translator(h)
        return translated

translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])
translators = translators.to("cuda")

total_params = sum(p.numel() for p in translators.parameters())
print(f"Tuned lens translators: {n_layers} layers")
print(f"Parameters per translator: {hidden_size * hidden_size + hidden_size:,}")
print(f"Total tuned lens parameters: {total_params:,}")

In [ ]:
N_TRAIN = 200

print(f"Collecting hidden states from {N_TRAIN} training samples...")

hook_storage = {"hidden_states": {}}
hooks = []

def make_hook(layer_idx):
    def hook(module, input, output):
        hook_storage["hidden_states"][layer_idx] = output[0].detach()
    return hook

for i, block in enumerate(model.transformer.h):
    hooks.append(block.register_forward_hook(make_hook(i)))

all_hidden_states = {i: [] for i in range(n_layers)}
all_final_logits = []

with torch.no_grad():
    for idx, sample in enumerate(tqdm(corpus[:N_TRAIN])):
        query = sample["query"]
        prompt = f"Question: {query}\nAnswer:"

        inputs = tokenizer(prompt, return_tensors="pt",
                          truncation=True, max_length=128).to("cuda")

        hook_storage["hidden_states"].clear()
        outputs = model(**inputs)

        final_logits = outputs.logits[0, -1, :].float()
        all_final_logits.append(final_logits.cpu())

        for layer_idx in range(n_layers):
            hs = hook_storage["hidden_states"][layer_idx][0, -1, :]
            all_hidden_states[layer_idx].append(hs.cpu())

for h in hooks:
    h.remove()

final_logits_tensor = torch.stack(all_final_logits)
hidden_tensors = {
    i: torch.stack(all_hidden_states[i]) for i in range(n_layers)
}

print(f"\nTraining data collected:")
print(f"Final logits shape: {final_logits_tensor.shape}")
print(f"Hidden states per layer shape: {hidden_tensors[0].shape}")

In [ ]:
translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])
translators = translators.to("cuda")
print("Translators reinitialized from identity.")

In [ ]:
N_EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 32

with torch.no_grad():
    final_probs = torch.softmax(final_logits_tensor, dim=-1)

print(f"Training tuned-lens on {N_TRAIN} samples, {N_EPOCHS} epochs per layer...")
print(f"This trains {n_layers} translators sequentially\n")

layer_losses = {}

for layer_idx in tqdm(range(n_layers), desc="Training layers"):
    hidden = hidden_tensors[layer_idx].to("cuda")
    targets = final_probs.to("cuda")

    translator = translators[layer_idx]
    optimizer = optim.Adam(translator.parameters(), lr=LR)

    best_loss = float('inf')

    for epoch in range(N_EPOCHS):
        epoch_loss = 0
        n_batches = 0

        perm = torch.randperm(N_TRAIN)
        hidden_shuffled = hidden[perm]
        targets_shuffled = targets[perm]

        for batch_start in range(0, N_TRAIN, BATCH_SIZE):
            batch_hidden = hidden_shuffled[batch_start:batch_start+BATCH_SIZE]
            batch_targets = targets_shuffled[batch_start:batch_start+BATCH_SIZE]

            optimizer.zero_grad()

            translated = translator(batch_hidden)

            with torch.no_grad():
                ln_weight = model.transformer.ln_f.weight.float()
                ln_bias = model.transformer.ln_f.bias.float()
                lm_weight = model.lm_head.weight.float()

            mean = translated.mean(-1, keepdim=True)
            std = translated.std(-1, keepdim=True)
            normalized = (translated - mean) / (std + 1e-5)
            normed = normalized * ln_weight + ln_bias

            logits = normed @ lm_weight.T
            probs = torch.softmax(logits, dim=-1)

            kl_loss = torch.nn.functional.kl_div(
                torch.log(probs + 1e-10),
                batch_targets,
                reduction='batchmean'
            )

            kl_loss.backward()
            optimizer.step()

            epoch_loss += kl_loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        if avg_loss < best_loss:
            best_loss = avg_loss

    layer_losses[layer_idx] = best_loss

    if layer_idx % 8 == 0:
        print(f"Layer {layer_idx:2d} trained | best loss: {best_loss:.6f}")

print(f"\nAll {n_layers} layers trained.")
print(f"Mean final loss: {np.mean(list(layer_losses.values())):.6f}")
print(f"Best layer: {min(layer_losses, key=layer_losses.get)} | "
      f"Worst layer: {max(layer_losses, key=layer_losses.get)}")

In [ ]:
print("saving tuned lens checkpoints")

for layer_idx in range(n_layers):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt")
    torch.save({
        "layer_idx": layer_idx,
        "state_dict": translators[layer_idx].state_dict(),
        "hidden_size": hidden_size,
        "loss": layer_losses[layer_idx]
    }, checkpoint_path)

summary = {
    "n_layers": n_layers,
    "hidden_size": hidden_size,
    "vocab_size": vocab_size,
    "n_train_samples": N_TRAIN,
    "n_epochs": N_EPOCHS,
    "layer_losses": {str(k): v for k, v in layer_losses.items()},
    "mean_loss": float(np.mean(list(layer_losses.values())))
}
with open(os.path.join(CHECKPOINT_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"Checkpoints saved to {CHECKPOINT_DIR}")
print(f"Files: {len(os.listdir(CHECKPOINT_DIR))}")

In [ ]:
print("verifying checkpoint loading")

test_translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])

for layer_idx in range(n_layers):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt")
    checkpoint = torch.load(checkpoint_path)
    test_translators[layer_idx].load_state_dict(checkpoint["state_dict"])

test_translators = test_translators.to("cuda")
test_translators.eval()

sample_hidden = hidden_tensors[0][:1].to("cuda")
with torch.no_grad():
    translated = test_translators[0](sample_hidden)
    normed = model.transformer.ln_f(translated.half())
    logits = model.lm_head(normed).float()
    probs = torch.softmax(logits, dim=-1)

print(f"Test translation output shape: {logits.shape}")
print(f"Probabilities sum to 1: {probs.sum().item():.6f}")
print(f"\nTuned-lens checkpoints: confirmed")
print(f"04_tuned_lens_training: complete")

In [ ]:
from google.colab import runtime
runtime.unassign()